# **Phase 1 — Inspect**
Understand the shape, structure, and quality of the raw data

In [2]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 50)

RAW_PATH = "/content/cityflo_bus_service_metro_cities.csv"
df = pd.read_csv(RAW_PATH)
df.head()

,trip_id,booking_id,customer_id,customer_name,gender,age,city,route_id,route_name,origin_stop,destination_stop,bus_number,bus_type,driver_id,driver_name,trip_date,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,distance_km,fare_inr,discount_inr,payment_mode,booking_channel,seat_number,trip_status,cancellation_reason,rating,occupancy_pct,weather,is_peak_hour,subscription_type,device_type,gps_enabled,complaint_raised
0,TRP000768,BK47984879,CUST01278,Shalini Verma,Female,21.0,Mumbai,RTMU005,Lower Parel - Mulund,Lower Parel,Mulund,MH23CF4441,AC Sleeper,DRV0080,Aarav Bhat,2024-09-21,20:25,20:25,20:58,20:58,10.9,153,0.0,UPI,Corporate Portal,D8,Completed,NaN,5.0,84.6,Haze,0,Corporate Plan,iOS,true,No
1,TRP002439,BK14257322,CUST01290,Kalpana Chopra,Male,24.0,Hyderabad,RTHY038,HITEC City - Kondapur,HITEC City,Kondapur,TS27CF8651,AC Seater,DRV0068,Sanjay Joshi,2024-08-08,14:50,15:12,15:29,15:51,14.1,127,0.0,Debit Card,Kiosk,D7,Delayed-Completed,NaN,NaN,28.3,Heavy Rain,False,Weekly Pass,Android,True,0
2,TRP000793,BK67236768,CUST01285,Sai Naidu,Female,18.0,Pune,RTPU012,Aundh - Magarpatta,Aundh,Magarpatta,MH24CF3646,AC Seater,DRV0001,Lakshmi Patel,2024-03-19,08:15,08:15,09:42,09:42,27.6,234,19.0,Cash,Corporate Portal,D11,Completed,NaN,3.0,15.4,Cloudy,1,Single Ride,Android,Yes,No
3,TRP002802,BK59659578,CUST00625,Priya Patel,Female,23.0,Mumbai,RTMU010,Ghatkopar - Lower Parel,Ghatkopar,Lower Parel,MH13CF5853,AC Sleeper,DRV0036,Siddharth Kumar,2024-12-29,17:45,17:45,18:57,NaN,28.6,310,0.0,Debit Card,Corporate Portal,D11,No-show,Customer Request,NaN,68.1,Clear,No,Corporate Plan,Web,1,Yes
4,TRP001653,BK19231279,CUST00276,Shalini Kulkarni,F,20.0,Bangalore,RTBA024,Hebbal - Electronic City,Hebbal,Electronic City,KA22CF6238,AC Seater,DRV0045,Sunita Gupta,2024-05-15,13:00,13:40,13:35,14:15,17.8,202,0.0,UPI,Corporate Portal,A1,Delayed-Completed,NaN,NaN,23.7,Haze,No,Corporate Plan,iOS,True,0


# **Step 1 — Total Number of Rows**
Check the row count to understand the dataset's size (df.shape[0]).

In [3]:
n_rows = df.shape[0]
print(f"Total rows: {n_rows}")

Total rows: 3258


# **Step 2 — Total Number of Columns**
Check the column count (df.shape[1]).

In [4]:
n_cols = df.shape[1]
print(f"Total columns: {n_cols}")

Total columns: 36


# **Step 3 — Understanding of Each Column**
Go through every column and note what it represents, its expected values, and how it relates to the problem.

In [5]:
print(df.columns)

Index(['trip_id', 'booking_id', 'customer_id', 'customer_name', 'gender',
       'age', 'city', 'route_id', 'route_name', 'origin_stop',
       'destination_stop', 'bus_number', 'bus_type', 'driver_id',
       'driver_name', 'trip_date', 'scheduled_departure', 'actual_departure',
       'scheduled_arrival', 'actual_arrival', 'distance_km', 'fare_inr',
       'discount_inr', 'payment_mode', 'booking_channel', 'seat_number',
       'trip_status', 'cancellation_reason', 'rating', 'occupancy_pct',
       'weather', 'is_peak_hour', 'subscription_type', 'device_type',
       'gps_enabled', 'complaint_raised'],
      dtype='object')


In [11]:
info_df = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "sample_value": df.iloc[0]
})
info_df

,dtype,n_unique,sample_value
trip_id,object,3200,TRP000768
booking_id,object,3200,BK47984879
customer_id,object,1251,CUST01278
customer_name,object,866,Shalini Verma
gender,object,11,Female
age,float64,40,21.0
city,object,12,Mumbai
route_id,object,60,RTMU005
route_name,object,60,Lower Parel - Mulund
origin_stop,object,40,Lower Parel


# **Step 4 — Trim Extra Spaces**
Strip leading/trailing whitespace from string columns and column headers — hidden spaces silently break groupby, filtering, and joins. We first detect which columns are affected before fixing them (the actual fix happens formally in Step 16, but we flag it here as required by Step 4).

In [12]:
# Column headers - check for stray whitespace
print("Header whitespace issues:", [c for c in df.columns if c != c.strip()])

# Detect string columns with leading/trailing whitespace
obj_cols = df.select_dtypes(include="object").columns
whitespace_flagged = {
    col: int((df[col].astype(str).str.strip() != df[col].astype(str)).sum())
    for col in obj_cols
}
{k: v for k, v in whitespace_flagged.items() if v > 0}

Header whitespace issues: []


{'customer_name': 122, 'gender': 783, 'city': 104}

# **Step 5 — Remove Duplicate Elements**
Identify and drop duplicate rows (df.duplicated(), df.drop_duplicates()).

In [13]:
n_dupes = df.duplicated().sum()
print(f"Fully duplicated rows: {n_dupes}")
df.loc[df.duplicated(keep=False)].sort_values("trip_id").head(6)

Fully duplicated rows: 58


,trip_id,booking_id,customer_id,customer_name,gender,age,city,route_id,route_name,origin_stop,destination_stop,bus_number,bus_type,driver_id,driver_name,trip_date,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,distance_km,fare_inr,discount_inr,payment_mode,booking_channel,seat_number,trip_status,cancellation_reason,rating,occupancy_pct,weather,is_peak_hour,subscription_type,device_type,gps_enabled,complaint_raised
2082,TRP000045,BK31856387,CUST00757,Ananya Menon,M,32.0,Pune,RTPU014,Shivajinagar - Aundh,Shivajinagar,Aundh,MH48CF6442,Premium AC,DRV0010,Sai Iyer,2024-12-12,20:20,NaN,20:41,NaN,9.3,0,0.0,NaN,Corporate Portal,D1,Cancelled,Driver Unavailable,NaN,NaN,Haze,Yes,Monthly Pass,Android,True,0
2771,TRP000045,BK31856387,CUST00757,Ananya Menon,M,32.0,Pune,RTPU014,Shivajinagar - Aundh,Shivajinagar,Aundh,MH48CF6442,Premium AC,DRV0010,Sai Iyer,2024-12-12,20:20,NaN,20:41,NaN,9.3,0,0.0,NaN,Corporate Portal,D1,Cancelled,Driver Unavailable,NaN,NaN,Haze,Yes,Monthly Pass,Android,True,0
2765,TRP000085,BK64561490,CUST00511,Sunita Malhotra,Female,25.0,Mumbai,RTMU004,Thane West - BKC,Thane West,BKC,MH16CF7658,Premium AC,DRV0048,Divya Rao,2024-12-13,20:10,20:10,21:53,21:53,41.7,752,0.0,Debit Card,Corporate Portal,A7,Completed,NaN,5.0,95.0,Haze,False,Weekly Pass,iOS,true,0
1668,TRP000085,BK64561490,CUST00511,Sunita Malhotra,Female,25.0,Mumbai,RTMU004,Thane West - BKC,Thane West,BKC,MH16CF7658,Premium AC,DRV0048,Divya Rao,2024-12-13,20:10,20:10,21:53,21:53,41.7,752,0.0,Debit Card,Corporate Portal,A7,Completed,NaN,5.0,95.0,Haze,False,Weekly Pass,iOS,true,0
2093,TRP000234,BK53828083,CUST00340,Karan Pillai,Male,19.0,Pune,RTPU013,Hadapsar - Kothrud,Hadapsar,Kothrud,MH17CF2891,AC Seater,DRV0065,Vikram Bansal,06/01/2024,09:10,09:37,09:29,09:56,9.1,97,0.0,Cash,Mobile App,B15,Delayed-Completed,NaN,5.0,68.6,Fog,0,Corporate Plan,Web,1,No
2799,TRP000234,BK53828083,CUST00340,Karan Pillai,Male,19.0,Pune,RTPU013,Hadapsar - Kothrud,Hadapsar,Kothrud,MH17CF2891,AC Seater,DRV0065,Vikram Bansal,06/01/2024,09:10,09:37,09:29,09:56,9.1,97,0.0,Cash,Mobile App,B15,Delayed-Completed,NaN,5.0,68.6,Fog,0,Corporate Plan,Web,1,No


# **Step 6 — Check Memory Size**
Check memory usage (df.memory_usage(deep=True)) — flags if dtypes need downcasting for efficiency.

In [14]:
mem = df.memory_usage(deep=True)
print(mem)
print(f"\nTotal memory: {mem.sum() / 1024**2:.2f} MB")

Index                     132
trip_id                188964
booking_id             192222
customer_id            188964
customer_name          198257
gender                 173487
age                     26064
city                   183804
route_id               182448
route_name             227747
origin_stop            188905
destination_stop       188710
bus_number             192222
bus_type               193935
driver_id              182448
driver_name            196754
trip_date              193857
scheduled_departure    175932
actual_departure       167572
scheduled_arrival      175932
actual_arrival         160092
distance_km             26064
fare_inr               180112
discount_inr            26064
payment_mode           174607
booking_channel        190617
seat_number            167407
trip_status            191404
cancellation_reason    127439
rating                  26064
occupancy_pct           26064
weather                176941
is_peak_hour           168351
subscripti

# **Step 7 — Check Data Type of Columns**
Verify each column's dtype matches what it should be (e.g., dates not stored as text, numbers not stored as objects).

In [15]:
df.dtypes

,0
trip_id,object
booking_id,object
customer_id,object
customer_name,object
gender,object
age,float64
city,object
route_id,object
route_name,object
origin_stop,object


In [16]:
# Columns that SHOULD be numeric/datetime but are currently loaded as object (text)
should_be_numeric = ["age", "distance_km", "fare_inr", "discount_inr", "rating", "occupancy_pct"]
should_be_datetime = ["trip_date"]
should_be_boolean = ["is_peak_hour", "gps_enabled", "complaint_raised"]

print("Currently wrong dtype (numeric expected):")
print(df[should_be_numeric].dtypes)
print("\nCurrently wrong dtype (datetime expected):")
print(df[should_be_datetime].dtypes)
print("\nCurrently wrong dtype (boolean expected):")
print(df[should_be_boolean].dtypes)

Currently wrong dtype (numeric expected):
age              float64
distance_km      float64
fare_inr          object
discount_inr     float64
rating           float64
occupancy_pct    float64
dtype: object

Currently wrong dtype (datetime expected):
trip_date    object
dtype: object

Currently wrong dtype (boolean expected):
is_peak_hour        object
gps_enabled         object
complaint_raised    object
dtype: object


# **Step 8 — Check Total Number of Null Values**
Count missing values per column (df.isnull().sum()) to plan the cleaning strategy.

In [17]:
null_counts = df.isnull().sum().sort_values(ascending=False)
null_pct = (null_counts / len(df) * 100).round(2)
pd.DataFrame({"nulls": null_counts, "pct_missing": null_pct}).loc[null_counts > 0]

,nulls,pct_missing
cancellation_reason,2538,77.90
rating,1075,33.00
actual_arrival,720,22.10
actual_departure,380,11.66
payment_mode,380,11.66
occupancy_pct,380,11.66
discount_inr,184,5.65
age,64,1.96
driver_name,32,0.98
